In [ ]:
# Install Tiny-SIMA into the active Jupyter kernel.
import os
import shutil
import subprocess
import sys
from pathlib import Path

os.environ.setdefault(
    "XLA_PYTHON_CLIENT_MEM_FRACTION",
    os.environ.get("TINY_SIMA_JAX_MEMORY_FRACTION", "0.60"),
)

if not ((3, 11) <= sys.version_info[:2] < (3, 13)):
    raise RuntimeError("Tiny-SIMA requires Python 3.11 or 3.12.")

repo_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pyproject.toml").is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError("Open this notebook from the Tiny-SIMA repository.")

cuda_override = os.environ.get("TINY_SIMA_USE_CUDA")
if cuda_override is None:
    has_cuda = (
        shutil.which("nvidia-smi") is not None
        and subprocess.run(
            ["nvidia-smi"],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
            check=False,
        ).returncode == 0
    )
else:
    has_cuda = cuda_override.lower() in {"1", "true", "yes"}

extras = "notebook,cuda12" if has_cuda else "notebook"
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--quiet",
    "--no-cache-dir",
    "--disable-pip-version-check",
    "-e",
    f"{repo_root}[{extras}]",
])

source_root = str(repo_root / "src")
if source_root not in sys.path:
    sys.path.insert(0, source_root)

print("Repository:", repo_root)
print("Source path added to kernel:", source_root)
print("CUDA detected:", has_cuda)


# Phase 4: local navigation and hierarchical skill chains

This notebook measures two generalization gaps before adding a VLM:

1. Can the executor solve randomized local-navigation scenarios with irrelevant objects?
2. Can skills be composed in one persistent Craftax world without resetting between requests?

The chain planner ranks each candidate by `utility × feasibility`, then replans after every success or failure. The scripted executor validates the environment and chain itself; the learned executor reveals the current policy ceiling.

In [ ]:
import jax
import matplotlib.pyplot as plt

from tiny_sima.envs import SkillScenario, make_skill_environment, render_state
from tiny_sima.evaluation import (
    ScriptedSkillExecutor,
    evaluate_goal_conditioned_checkpoint,
    evaluate_skill_chain,
    evaluate_skill_suite,
    learned_skill_executor_from_checkpoint,
)
from tiny_sima.goals import SkillId

print("JAX devices:", jax.devices())


In [ ]:
CHECKPOINT = repo_root / "runs" / "skill-ppo-1m" / "checkpoints" / "final"
NUM_LOCAL_EPISODES = 128
NUM_CHAIN_EPISODES = 8
LOCAL_HORIZON = 32
CHAIN_SKILL_HORIZON = 64

print("Checkpoint:", CHECKPOINT)
print("Checkpoint exists:", CHECKPOINT.exists())


In [ ]:
local_environment = make_skill_environment(
    SkillId.COLLECT_WOOD,
    horizon=LOCAL_HORIZON,
    scenario=SkillScenario.LOCAL_NAVIGATION,
    min_distance=2,
    max_distance=4,
    distractor_count=6,
)
local_reset = local_environment.reset(jax.random.PRNGKey(7))
pixels = render_state(
    local_reset.state.env_state,
    local_environment.bundle.spec.name,
    block_pixel_size=16,
)

plt.figure(figsize=(7, 5))
plt.imshow(pixels)
plt.axis("off")
plt.title("Randomized local-navigation scenario")
plt.show()


In [ ]:
# The scripted shortest-path policy should remain at 100%; otherwise the
# scenario generator is producing malformed or unreachable episodes.
scripted_local = evaluate_skill_suite(
    num_episodes=32,
    horizon=LOCAL_HORIZON,
    seed=11,
    scenario=SkillScenario.LOCAL_NAVIGATION,
    min_distance=2,
    max_distance=4,
    distractor_count=6,
)
for result in scripted_local:
    if result["policy"] == "scripted":
        print(
            f'{result["skill_id"]:22s} '
            f'scripted={result["success_rate"]:.3f} '
            f'length={result["mean_episode_length"]:.2f}'
        )


In [ ]:
if CHECKPOINT.exists():
    learned_local = evaluate_goal_conditioned_checkpoint(
        CHECKPOINT,
        num_episodes_per_skill=NUM_LOCAL_EPISODES,
        seed=21,
        deterministic=True,
        scenario=SkillScenario.LOCAL_NAVIGATION,
        horizon=LOCAL_HORIZON,
        min_distance=2,
        max_distance=4,
        distractor_count=6,
    )
    for result in learned_local:
        print(
            f'{result["skill_id"]:22s} '
            f'success={result["success_rate"]:.3f} '
            f'timeout={result["feasible_timeout_rate"]:.3f} '
            f'length={result["mean_episode_length"]:.2f}'
        )
else:
    print("Checkpoint not found; skipping learned local-navigation evaluation.")


In [ ]:
# This validates persistent state, chain preconditions, outcome accounting,
# and closed-loop replanning independently of the learned policy.
oracle_chain = evaluate_skill_chain(
    ScriptedSkillExecutor(),
    num_episodes=NUM_CHAIN_EPISODES,
    seed=31,
    skill_horizon=CHAIN_SKILL_HORIZON,
)
{key: value for key, value in oracle_chain.items() if key != "episodes"}


In [ ]:
if CHECKPOINT.exists():
    learned_executor = learned_skill_executor_from_checkpoint(
        CHECKPOINT,
        deterministic=True,
    )
    learned_chain = evaluate_skill_chain(
        learned_executor,
        num_episodes=NUM_CHAIN_EPISODES,
        seed=41,
        skill_horizon=CHAIN_SKILL_HORIZON,
    )
    print({key: value for key, value in learned_chain.items() if key != "episodes"})
    print("\nFirst episode decision trace:")
    for trace in learned_chain["episodes"][0].trace:
        print(
            f'{trace.decision_index:2d} requested={trace.requested_skill.value:20s} '
            f'outcome={trace.outcome.value:24s} steps={trace.steps:2d} '
            f'utility={trace.utility:.2f} feasibility={trace.feasibility:.2f} '
            f'score={trace.combined_score:.2f}'
        )
else:
    print("Checkpoint not found; skipping learned chain evaluation.")


## How to interpret the comparison

- Scripted local failure means the scenario generator is malformed.
- Learned local failure isolates navigation or distractor generalization.
- Scripted chain failure means the persistent chain or preconditions are wrong.
- Learned chain failure with a successful scripted chain measures executor compounding.

The future VLM replaces only the utility scorer. Symbolic or learned feasibility remains a separate factor, and every skill outcome is returned to the planner before it makes the next decision.